# Avance 4 — Modelos alternativos

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Compara familias de modelos **individuales** sobre las dos fases del proyecto, ajusta las dos
mejores y elige una. Parte del nivel de referencia del [Avance 3](03_Baseline.html) y usa la
misma partición agrupada y las mismas métricas, para que las cifras sean comparables.

## Qué produce

1. Una **comparativa de seis familias sin ensamble**, ordenada por la métrica principal, con
   otras métricas y los tiempos de entrenamiento.
2. El **ajuste de hiperparámetros** de las dos mejores, con la búsqueda **dentro** del pliegue.
3. La **estabilidad** de lo que esa búsqueda eligió entre pliegues.
4. El **modelo individual final** y el argumento de por qué ese.

## Por qué no aparecen aquí el bosque ni el boosting

La rúbrica pide seis modelos «individuales, no ensambles», y la del Avance 5 pide ensambles
homogéneos y heterogéneos. Un bosque aleatorio y un boosting son ensambles homogéneos por
construcción, así que pertenecen al Avance 5. Se incluyen en la tabla como referencia del
Avance 3, marcados y fuera del conteo de seis.

## 1. Preparación

In [ ]:
import os
# Un solo hilo por modelo: así la columna de tiempos significa lo mismo en todas las filas.
os.environ["OMP_NUM_THREADS"] = "1"

import platform
import warnings
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score,
                             matthews_corrcoef, precision_recall_curve, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Este notebook se ejecuta desde `notebooks/` o desde la raíz del repositorio.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESADOS = RAIZ / "data/processed"

In [ ]:
# Versiones con las que se ejecutó. Las cifras cambian en el tercer decimal entre versiones,
# así que sin esto una salida no se puede rastrear a su entorno. Tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn):
    print(f"{m.__name__:<8} {m.__version__}")

### Tres funciones cortas

Todo lo demás es scikit-learn directo. Estas tres son lo único propio del notebook:

- **`grupos_cv`**: junta andamio y esqueleto de conectividad en un solo grupo. Es lo que impide que
  dos análogos del mismo compuesto caigan en pliegues distintos. Sin esto las cifras se inflan entre
  7 y 13 puntos (medido en la fase 1: agrupar solo por andamio da
  2.2 puntos más de AUROC y 7.1 más de MCC).
- **`medir`**: MCC, AUROC y AUPRC de la clase minoritaria. El MCC se calcula con la etiqueta que
  predice el modelo, y las otras dos con su puntaje.
- **`evaluar`**: validación cruzada agrupada, tres semillas y cinco pliegues. Si recibe una rejilla,
  elige los hiperparámetros **dentro** del pliegue, con una validación interna que también agrupa.

In [ ]:
SIN_ANDAMIO = "<aciclico>"   # los compuestos sin anillos no tienen andamio


def grupos_cv(andamio, esqueleto):
    """Grupo = esqueleto de conectividad, unido al andamio cuando lo hay."""
    padre = {}

    def raiz(x):
        while padre.setdefault(x, x) != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    for a, e in zip(andamio, esqueleto):
        if a != SIN_ANDAMIO:
            padre[raiz("A" + a)] = raiz("E" + e)
    return np.array([raiz("E" + e) for e in esqueleto])


def medir(y, etiqueta, puntaje):
    minoritaria = y.mean() <= 0.5          # cuál es la minoritaria cambia entre fases
    ym, pm = (y, puntaje) if minoritaria else (1 - y, -puntaje)
    return {"mcc": matthews_corrcoef(y, etiqueta),
            "auroc": roc_auc_score(y, puntaje),
            "auprc_minoritaria": average_precision_score(ym, pm)}


def evaluar(modelos, X, y, grupos, estratos=None, semillas=(0, 1, 2), pliegues=5, rejillas=None):
    """Una fila por (modelo, semilla, pliegue, estrato) y las predicciones fuera de pliegue."""
    filas = []
    fuera = {n: (np.full((len(semillas), len(y)), np.nan),
                 np.full((len(semillas), len(y)), -1)) for n in modelos}
    for s, semilla in enumerate(semillas):
        cv = StratifiedGroupKFold(pliegues, shuffle=True, random_state=semilla)
        for k, (ajuste, val) in enumerate(cv.split(X, y, grupos), 1):
            for nombre, modelo in modelos.items():
                m = clone(modelo)
                # La semilla cambia la partición y también la del modelo.
                m.set_params(**{p: semilla for p in m.get_params() if p.endswith("random_state")})
                if rejillas and nombre in rejillas:
                    m = GridSearchCV(m, rejillas[nombre], scoring="matthews_corrcoef", n_jobs=1,
                                     cv=StratifiedGroupKFold(3, shuffle=True, random_state=semilla))
                    reloj = perf_counter()
                    m.fit(X[ajuste], y[ajuste], groups=grupos[ajuste])
                else:
                    reloj = perf_counter()
                    m.fit(X[ajuste], y[ajuste])
                seg = perf_counter() - reloj
                puntaje = (m.predict_proba(X[val])[:, 1] if hasattr(m, "predict_proba")
                           else m.decision_function(X[val]))
                etiqueta = m.predict(X[val])
                fuera[nombre][0][s, val], fuera[nombre][1][s, val] = puntaje, etiqueta
                fila = dict(modelo=nombre, semilla=semilla, pliegue=k, seg=seg)
                if hasattr(m, "best_params_"):
                    fila["elegidos"] = str(m.best_params_)
                filas.append({**fila, "estrato": "todos", **medir(y[val], etiqueta, puntaje)})
                if estratos is None:
                    continue
                for e in np.unique(estratos[val]):
                    sel = estratos[val] == e
                    ys = y[val][sel]
                    if min(ys.sum(), (1 - ys).sum()) >= 20:      # estrato con 20 casos de cada clase
                        filas.append({**fila, "estrato": e,
                                      **medir(ys, etiqueta[sel], puntaje[sel])})
    return pd.DataFrame(filas), fuera


def tabla(res, estrato="todos"):
    """Media por modelo, ordenada por MCC. `desv_mcc` es dispersión entre pliegues, no error estándar."""
    d = res[res["estrato"] == estrato]
    t = d.groupby("modelo")[["mcc", "auroc", "auprc_minoritaria", "seg"]].mean()
    t.insert(1, "desv_mcc", d.groupby("modelo")["mcc"].std())
    return t.sort_values("mcc", ascending=False).round(3)

### Los datos

Salen del [notebook 02](02_Feature_Eng.html) (fase 1) y del [02b](02b_Feature_Eng_actividad.html)
(fase 2), que dejan sus matrices en `data/processed/`. Ese directorio no se versiona, así que hay que
haber ejecutado esos dos notebooks antes. Si falta un archivo, la celda lo dice.

- **Fase 1, permeabilidad:** 151 descriptores de RDKit sin transformar (el escalado se hace dentro de
  cada modelo, para que no vea el pliegue de evaluación) y la etiqueta BBB+ / BBB−.
- **La prueba reservada queda fuera.** Aquí solo se usan entrenamiento y validación.

In [ ]:
for f in ("bbb_meta.parquet", "bbb_descriptores_filtrados_sin_normalizar.parquet",
          "fase2_pares.parquet", "fase2_descriptores_limpios.parquet"):
    if not (PROCESADOS / f).exists():
        raise FileNotFoundError(f"Falta data/processed/{f}. Lo genera el notebook 02 (bbb_*) "
                                "o el 02b (fase2_*); ejecútenlos primero.")

meta = pd.read_parquet(PROCESADOS / "bbb_meta.parquet")
descriptores = pd.read_parquet(PROCESADOS / "bbb_descriptores_filtrados_sin_normalizar.parquet")

uso = meta["particion"].ne("prueba").to_numpy()               # la prueba no se toca
m1 = meta[uso].reset_index(drop=True)
X1 = descriptores.to_numpy(dtype="float32")[uso]
y1 = m1["y"].astype(int).to_numpy()                           # BBB+ = 1
g1 = grupos_cv(m1["andamio"].fillna(SIN_ANDAMIO), m1["esqueleto"])
print(f"{X1.shape[0]:,} compuestos · {X1.shape[1]} descriptores · {len(np.unique(g1)):,} grupos "
      f"· {y1.mean():.1%} BBB+")

## 2. Las seis familias

Seis algoritmos **sin ensamble** con sesgos distintos: un modelo lineal (la logística, el control
interpretable), un árbol, uno basado en vecinos, uno generativo (Naive Bayes), uno de núcleo
(SVM con RBF) y una red pequeña. La rúbrica evalúa que sean variados, no que sean los seis de
mayor métrica: el árbol y Naive Bayes entran aunque queden últimos. Se descartaron LDA, por
parecerse a la logística, y el SVM lineal por SGD, porque el SVM con RBF ya cubre esa familia.
El bosque y el boosting son ensambles y van al Avance 5.

In [ ]:
SEIS = ("logistica", "arbol", "knn", "bayes", "svm_rbf", "mlp")
PRINCIPAL = "mcc"          # la métrica que ordena la tabla; vale 0 para un modelo trivial

# Los que necesitan escala la llevan dentro, para ajustarla solo con el pliegue de entrenamiento.
modelos = {
    "logistica": make_pipeline(MinMaxScaler(), LogisticRegression(max_iter=3000)),
    "arbol":     DecisionTreeClassifier(min_samples_leaf=5),
    "knn":       make_pipeline(MinMaxScaler(), KNeighborsClassifier(n_neighbors=15)),
    "bayes":     make_pipeline(MinMaxScaler(), GaussianNB()),
    "svm_rbf":   make_pipeline(MinMaxScaler(), SVC()),
    "mlp":       make_pipeline(MinMaxScaler(), MLPClassifier(hidden_layer_sizes=(64,), max_iter=600)),
}
assert tuple(modelos) == SEIS

## 3. Comparativa

Las seis familias con partición agrupada por andamio y esqueleto, tres semillas y cinco pliegues,
como en el Avance 3. Los tiempos son de reloj de pared, así que para que la columna
signifique lo mismo en todas las filas hay que correr con el mismo paralelismo.

In [ ]:
res1, fdp1 = evaluar(modelos, X1, y1, g1)
tabla1 = tabla(res1)
tabla1

> **Pendiente del equipo.** Leer la tabla de la fase 1. Qué familia encabeza, si la distancia a la segunda es mayor que la dispersión, y qué cuesta en tiempo.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

In [ ]:
# Los dos ensambles del Avance 3, como referencia y fuera del conteo de seis.
referencia = {
    "bosque (A3)": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=1),
    "boosting (A3)": HistGradientBoostingClassifier(),
}
ref1, _ = evaluar(referencia, X1, y1, g1)
tabla(ref1)

> **Pendiente del equipo.** Comparar las seis familias individuales contra los dos ensambles del Avance 3, sin contarlos entre los seis.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

### 3.1 Fase 2 · Actividad en el sistema nervioso

La fase 2 tiene 54,369 pares y la unidad es el par compuesto-diana. Solo se corren **los dos de mayor
AUROC en la tabla anterior**, regla fijada antes de ver la fase 2. Un SVM con RBF crece entre
cuadrático y cúbico: un ajuste sobre el pliegue completo ronda los cinco minutos, así que si es uno de
los dos conviene correrlo con una sola semilla (`semillas=(0,)`) y declararlo.

Los dos se eligen con la comparación sin afinar de la fase 1 y luego se afinan en la §4. El ajuste
es anidado, así que sus cifras son honestas; lo que queda es que la elección de los dos usó esa
misma comparación. Es un sesgo pequeño y se declara.

In [ ]:
DOS = list(tabla1.sort_values("auroc", ascending=False).index[:2])
print("los dos de mayor AUROC:", DOS)

pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc2 = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d2 = (pares[pares["activo_10um"].notna() & pares["particion"].ne("prueba")]
      .merge(desc2, on="smiles").reset_index(drop=True))
cols = [c for c in desc2.columns if c != "smiles"]
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se mide.
X2 = np.hstack([d2[cols].to_numpy("float32"), pd.get_dummies(d2["target_gene"]).to_numpy("float32")])
y2 = d2["activo_10um"].astype(int).to_numpy()
g2 = d2["grupo_cv"].to_numpy()
procedencia = d2["estrato"].to_numpy()              # solo_chembl, solo_npass, mixto

res2, fdp2 = evaluar({n: modelos[n] for n in DOS}, X2, y2, g2, estratos=procedencia)
tabla(res2)

> **Pendiente del equipo.** Leer la fase 2, y el desglose por procedencia: un agregado no describe a los productos naturales (en el control del Avance 3, el AUROC baja de 0.83 a 0.57 en los pares que solo tiene NPASS).
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

## 4. Ajuste fino de las dos mejores

La búsqueda va **dentro** del pliegue y su validación interna también agrupa. Hacerlo con una
partición aleatoria dejaría la evaluación externa intacta y elegiría la rejilla sobre una
medición inflada entre 7 y 13 puntos, que es una fuga que no aparece en ninguna métrica.

In [ ]:
# Rejillas de las seis; solo se usan las de los dos elegidos. Búsqueda exhaustiva en todas.
# Los nombres son los que make_pipeline le da a cada paso (minúsculas del nombre de la clase).
REJILLAS = {
    "logistica": {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
    "arbol":     {"max_depth": [3, 6, 10, None], "min_samples_leaf": [1, 5, 20]},
    "knn":       {"kneighborsclassifier__n_neighbors": [5, 15, 30, 50],
                  "kneighborsclassifier__weights": ["uniform", "distance"]},
    "bayes":     {"gaussiannb__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3]},
    "svm_rbf":   {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", "auto"]},
    "mlp":       {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (128,)],
                  "mlpclassifier__alpha": [1e-5, 1e-4, 1e-3, 1e-2]},
}

afinados = {f"{n} afinado": modelos[n] for n in DOS}
rejillas = {f"{n} afinado": REJILLAS[n] for n in DOS}
res_af, fdp_af = evaluar(afinados, X1, y1, g1, rejillas=rejillas)
tabla(res_af)

> **Pendiente del equipo.** Decir si el ajuste mejoró, cuánto, y si la mejora es mayor que la dispersión entre pliegues.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

### 4.1 Qué eligió la búsqueda en cada pliegue

Una rejilla que cambia de pliegue a pliegue no es un óptimo: es una elección que depende de
la partición. Es la diferencia entre un ajuste fino y una corrida afortunada.

In [ ]:
res_af[["modelo", "semilla", "pliegue", PRINCIPAL, "elegidos"]].to_string(index=False)

> **Pendiente del equipo.** Decir si lo elegido fue estable entre pliegues o cambió.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

## 5. Gráficos del mejor individual

Sobre predicciones **fuera de pliegue**, no de un modelo reajustado sobre todo el conjunto:
un gráfico hecho sobre datos que el modelo vio describe su memoria, no su desempeño.

In [ ]:
MEJOR = tabla1.index[0]
puntaje, etiqueta = fdp1[MEJOR]
p, e = puntaje[0], etiqueta[0].astype(int)              # semilla 0

fig, ax = plt.subplots(2, 2, figsize=(11, 9))

fpr, tpr, _ = roc_curve(y1, p)
ax[0, 0].plot(fpr, tpr, label=f"{MEJOR} · AUROC {roc_auc_score(y1, p):.3f}")
ax[0, 0].plot([0, 1], [0, 1], "--", color="0.6", label="azar")
ax[0, 0].set(xlabel="falsos positivos", ylabel="verdaderos positivos", title="Curva ROC")
ax[0, 0].legend()

prec, rec, _ = precision_recall_curve(1 - y1, -p)       # la minoritaria es BBB−
ax[0, 1].plot(rec, prec)
ax[0, 1].axhline((1 - y1).mean(), ls="--", color="0.6")
ax[0, 1].set(xlabel="exhaustividad", ylabel="precisión", title="Precisión-exhaustividad de BBB−")

ConfusionMatrixDisplay.from_predictions(y1, e, display_labels=["BBB−", "BBB+"], ax=ax[1, 0],
                                        colorbar=False)
ax[1, 0].set_title("Matriz de confusión")

# Importancia por permutación en un pliegue que el modelo no vio.
ajuste, val = next(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X1, y1, g1))
mod = clone(modelos[MEJOR]).fit(X1[ajuste], y1[ajuste])
imp = permutation_importance(mod, X1[val], y1[val], scoring="roc_auc", n_repeats=5, random_state=0)
top = pd.Series(imp.importances_mean, index=descriptores.columns).nlargest(15).iloc[::-1]
ax[1, 1].barh(top.index, top.values)
ax[1, 1].set(xlabel="caída del AUROC al permutar", title="Importancia por permutación")

fig.tight_layout()

> **Pendiente del equipo.** Interpretar los cuatro gráficos, uno por uno. La rúbrica del Avance 5 pide interpretación detallada, no la figura sola.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

## 6. El modelo individual final

La rúbrica da el máximo cuando el argumento no es solo la métrica: compensaciones entre
métricas, interpretabilidad, costo de entrenamiento, desempeño sobre los productos naturales.
Si los intervalos se solapan y ninguna familia se distingue, decirlo es el resultado.

> **Pendiente del equipo.** Escribir el argumento del modelo final.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

## 7. Qué queda establecido y qué no

> **Pendiente del equipo.** Cerrar el avance: qué se puede afirmar con la evidencia de esta libreta y qué no.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | | | |
| | | | |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | |
| Artemio Santiago Padilla Robles | |
| José Antonio Vázquez Martínez | |

### Decisiones principales

**Decisión 1:**

- Justificación:
- Alternativa descartada y motivo:

**Decisión 2:**

- Justificación:
- Alternativa descartada y motivo:

**Decisión 3:**

- Justificación:
- Alternativa descartada y motivo:

### Responsabilidad

Las salidas de las herramientas se usaron como insumo de trabajo y fueron revisadas,
verificadas, editadas y adaptadas por el equipo. El equipo asume la responsabilidad
por la exactitud, la integridad y el contenido final del avance.